# Exploration & Re-schema

เน้น **กระบวนการ/ขั้นตอนการทดลอง (process)** ในการทำ perovskite solar cell
(ต่างจากงานที่เน้นค้นหาวัสดุ/ธาตุใหม่). Raw data อยู่ในโฟลเดอร์ `Raw Data/` โดย
**1 ไฟล์ `.xlsx` = 1 device (sample)** ตั้งชื่อว่า `<doi>_Sample<N>.xlsx`.

แต่ละ workbook มี 6 sheets: `Layer`, `Vender`, `Materials`, `Process`, `Properties`, `Stability`.

## Setup

In [50]:
import sys, os, glob, re
import pandas as pd

# chdir ไป repo root เพื่อให้ relative path ("Raw Data", "processed") ใช้ได้เสมอ
# ไม่ว่าจะรัน notebook จากที่ไหน
ROOT = os.path.abspath(os.path.join(os.getcwd(), ".." if os.path.basename(os.getcwd())=="notebooks" else "."))
os.chdir(ROOT)

RAW_DIR = "Raw Data"
OUT_DIR = "processed"
files = sorted(glob.glob(os.path.join(RAW_DIR, "*.xlsx")))
print("repo root :", ROOT)
print("workbooks :", len(files))

repo root : d:\University\Year 4 Semester 1 (Intern)\Perovskites-Research
workbooks : 44


In [51]:
def parse_device_id(path: str):
    """filename -> (device_id, doi_raw, sample_no)."""
    stem = os.path.splitext(os.path.basename(path))[0]
    m = re.search(r"[_.]?[Ss]ample\s*(\d+)$", stem)
    sample_no = int(m.group(1)) if m else None
    doi_raw = stem[: m.start()] if m else stem
    return stem, doi_raw, sample_no

## Inventory — ไฟล์ / publication / sample

ชื่อไฟล์เข้ารหัส DOI + หมายเลข sample. แยกออกมาเป็นคีย์ที่ใช้งานได้.

In [52]:
inv = pd.DataFrame(
    [dict(zip(["device_id","doi_raw","sample_no"], parse_device_id(f))) | {"file": os.path.basename(f)}
     for f in files]
)
print(f"{len(inv)} devices จาก {inv['doi_raw'].nunique()} publications\n")
display(inv.head(10))

44 devices จาก 15 publications



,device_id,doi_raw,sample_no,file
0,10.1002admi.202500231_Sample1,10.1002admi.202500231,1,10.1002admi.202500231_Sample1.xlsx
1,10.1002admi.202500231_Sample2,10.1002admi.202500231,2,10.1002admi.202500231_Sample2.xlsx
2,10.1002admi.202500231_Sample3,10.1002admi.202500231,3,10.1002admi.202500231_Sample3.xlsx
3,10.1002admi.202500231_Sample4,10.1002admi.202500231,4,10.1002admi.202500231_Sample4.xlsx
4,10.1016j.jechem.2021.11.029_Sample1,10.1016j.jechem.2021.11.029,1,10.1016j.jechem.2021.11.029_Sample1.xlsx
5,10.1016j.jechem.2021.11.029_Sample2,10.1016j.jechem.2021.11.029,2,10.1016j.jechem.2021.11.029_Sample2.xlsx
6,10.1016j.nanoen.2021.106285_Sample1,10.1016j.nanoen.2021.106285,1,10.1016j.nanoen.2021.106285_Sample1.xlsx
7,10.1016j.nanoen.2021.106285_Sample2,10.1016j.nanoen.2021.106285,2,10.1016j.nanoen.2021.106285_Sample2.xlsx
8,10.1016j.nanoen.2021.106285_Sample3,10.1016j.nanoen.2021.106285,3,10.1016j.nanoen.2021.106285_Sample3.xlsx
9,10.1021_acsami.5c16972_Sample1,10.1021_acsami.5c16972,1,10.1021_acsami.5c16972_Sample1.xlsx


In [53]:
display(inv.groupby("doi_raw").size().rename("n_samples").sort_values(ascending=False))

doi_raw
10_1002adma_202507730             5
10_1016_j_heliyon_2024_e25352     5
10.1002admi.202500231             4
10_1039_d3lf00258f                4
10_1021_acsami_5c15868            3
10.1016j.nanoen.2021.106285       3
10.1021_acsami.5c16972            3
10_31635_ccschem_025_202505977    3
10_1002_solr_202200422            2
10_1002_smll_202504237            2
10.1016j.jechem.2021.11.029       2
10_1002_solr_202400534            2
10_1016_j_solener_2025_113286     2
10_1021_acsami_4c14300            2
10_1126_science_ade9637           2
Name: n_samples, dtype: int64

## Schema audit — sheet และ header สม่ำเสมอแค่ไหน?

ตรวจว่าทุกไฟล์มี sheet/หัวคอลัมน์ตรงกันหรือไม่ (จุดที่ต้อง normalize).

In [54]:
from collections import Counter

sheet_combos, headers = Counter(), {}
for f in files:
    xl = pd.ExcelFile(f)
    sheet_combos[tuple(xl.sheet_names)] += 1
    for sh in xl.sheet_names:
        h = tuple(str(x).strip() for x in pd.read_excel(f, sh, nrows=0).columns)
        headers.setdefault(sh, Counter())[h] += 1

print("=== ชุดชื่อ sheet ที่พบ ===")
for combo, n in sheet_combos.most_common():
    print(f"  {n:2d}x  {combo}")

=== ชุดชื่อ sheet ที่พบ ===
  32x  ('Layer', 'Vender', 'Materials', 'Process', 'Properties', 'Stability')
   8x  ('Layer', 'Materials', 'Vender', 'Process', 'Properties', 'Stability')
   4x  ('Layer', 'Vender', 'Materials', 'Process', 'NiOx synthesis', 'Properties', 'Stability')


In [55]:
print("\n=== จำนวน header แบบที่ต่างกัน ต่อ sheet ===")
for sh, c in headers.items():
    print(f"  {sh:12s}: {len(c)} แบบ")


=== จำนวน header แบบที่ต่างกัน ต่อ sheet ===
  Layer       : 1 แบบ
  Vender      : 1 แบบ
  Materials   : 1 แบบ
  Process     : 1 แบบ
  NiOx synthesis: 1 แบบ
  Properties  : 2 แบบ
  Stability   : 8 แบบ


In [56]:
for h, n in headers["Properties"].most_common():
    print(f"{n:2d}x {h}")

31x ('Properties', 'Values', 'Errors', 'Unit', 'Remarks')
13x ('Properties', 'Values', 'Unit', 'Remarks')


In [57]:
for h, n in headers["Stability"].most_common():
    print(f"{n:2d}x {h}")

27x ('Environment', 'Encapsulation', 'Temperature (°C)', 'Humidity (%)', 'Time (hr)', 'Retention (%)', 'T80 (hr)')
 4x ('Environment', 'Encapsulation', 'temperature (celsius)', 'relative humidity (%)', 'time (hr)', 'Normalize PCE (%)', 'Initial PCE (%)', 'Final PCE (%)')
 4x ('Environment', 'Encapsulation', 'temperature (C)', 'humidity (%)', 'time (hr)', 'Retention (%)')
 4x ('Environment', 'Encapsulation', 'Temperature (°C)', 'humidity (%)', 'time (hr)', 'Retention (%)')
 2x ('Environment', 'Encapsulation', 'temperature (celsius)', 'relative humidity (%)', 'time (hr)', 'Normalize PCE (%)')
 1x ('Environment', 'Encapsulation', 'temperature (C)', 'humidity (%)', 'time (hr)', 'Normalize PCE (%)')
 1x ('Environment', 'Encapsulation', 'temperature (celsius)', 'relative humidity (%)', 'time (hr)', 'Final normalize PCE (%)')
 1x ('Environment', 'Encapsulation', 'Temperature (°C)', 'Humidity (%)', 'Time (hr)', 'Retention (%)')


In [58]:
# Stability headers vary in casing/wording across papers; map them to one set.
STABILITY_MAP = {
    "environment": "environment",
    "encapsulation": "encapsulation",
    "temperature (°c)": "temperature_c",
    "temperature (c)": "temperature_c",
    "temperature (celsius)": "temperature_c",
    "humidity (%)": "humidity_pct",
    "relative humidity (%)": "humidity_pct",
    "time (hr)": "time_hr",
    "retention (%)": "retention_pct",
    "normalize pce (%)": "retention_pct",
    "final normalize pce (%)": "retention_pct",
    "t80 (hr)": "t80_hr",
    "initial pce (%)": "initial_pce",
    "final pce (%)": "final_pce",
}

STABILITY_COLS = ["environment", "encapsulation", "temperature_c",
                  "humidity_pct", "time_hr", "retention_pct", "t80_hr",
                  "initial_pce", "final_pce"]

In [59]:
# The 6 sheets every workbook is expected to have. Any *extra* sheet whose
# header looks like Process steps (has StepOrder + ProcessType) is treated as
# additional process/synthesis steps and folded into the `process` table
# (this project is process-centric). Anything else triggers a drift warning
# so new data is never silently dropped.
KNOWN_SHEETS = {"Layer", "Vender", "Materials", "Process",
                "Properties", "Stability"}

def _clean(df: pd.DataFrame) -> pd.DataFrame:
    df.columns = [str(c).strip() for c in df.columns]
    # normalize whitespace/newlines inside string cells.
    # NB: check for string OR object dtype — pandas >=3.0 reads text columns as
    # the new "str" dtype (not "object"), so an `== object` test silently skips
    # all cleaning and leaves untrimmed whitespace (e.g. "NiOx " vs "NiOx").
    for c in df.columns:
        if pd.api.types.is_string_dtype(df[c]) or df[c].dtype == object:
            df[c] = (df[c].astype(str)
                          .str.replace(r"\s+", " ", regex=True)
                          .str.strip()
                          .replace({"nan": pd.NA, "": pd.NA}))
    return df

def _is_process_like(cols) -> bool:
    s = {str(c).strip() for c in cols}
    return {"StepOrder", "ProcessType"} <= s

In [60]:
sample_file = files[0]
print("FILE:", os.path.basename(sample_file), "\n")
for sh in ["Process"]:
    print(f"───── {sh} ─────")
    display(pd.read_excel(sample_file, sh))

FILE: 10.1002admi.202500231_Sample1.xlsx 

───── Process ─────


,LayerID,StepOrder,ProcessType,Descriptions,Parameter,Unit
0,1,1,Wash,"TritonX solution, Sonication, Acetone, Isopro...",1200,sec
1,1,2,PlasmaTreat,O2,155600600,"W, mtorr, sec"
2,2,1,Sonication,NaN,3600,sec
3,2,2,SpinCoating,NaN,"3000, 30","RPM, sec"
4,2,3,Annealing,NaN,"270, 10800","C, sec"
5,4,1,Magnetic Stir,NaN,"60, 3600","C, sec"
6,4,2,SpinCoating,NaN,"1000,500,10","RPM, RPM/s,sec"
7,4,3,SpinCoating,NaN,"4000,2000,35","RPM,RPM/s,sec"
8,4,4,AntiSolvent,CB,"0.25, -10","ml, sec"
9,4,5,Annealing,NaN,"100, 1800","C, sec"


### สำหรับไฟล์ที่มี NiOx synthesis เพิ่มเข้ามา

In [61]:
target = None
for f in files:
    xl = pd.ExcelFile(f)
    if "NiOx synthesis" in xl.sheet_names:
        target = f
        break

print("FILE:", os.path.basename(target))
print("sheets:", xl.sheet_names)

print("\n===== Process (raw, as-is) =====")
display(pd.read_excel(target, "Process"))

print("\n===== NiOx synthesis (raw, as-is) =====")
display(pd.read_excel(target, "NiOx synthesis"))


FILE: 10.1002admi.202500231_Sample1.xlsx
sheets: ['Layer', 'Vender', 'Materials', 'Process', 'NiOx synthesis', 'Properties', 'Stability']

===== Process (raw, as-is) =====


,LayerID,StepOrder,ProcessType,Descriptions,Parameter,Unit
0,1,1,Wash,"TritonX solution, Sonication, Acetone, Isopro...",1200,sec
1,1,2,PlasmaTreat,O2,155600600,"W, mtorr, sec"
2,2,1,Sonication,NaN,3600,sec
3,2,2,SpinCoating,NaN,"3000, 30","RPM, sec"
4,2,3,Annealing,NaN,"270, 10800","C, sec"
5,4,1,Magnetic Stir,NaN,"60, 3600","C, sec"
6,4,2,SpinCoating,NaN,"1000,500,10","RPM, RPM/s,sec"
7,4,3,SpinCoating,NaN,"4000,2000,35","RPM,RPM/s,sec"
8,4,4,AntiSolvent,CB,"0.25, -10","ml, sec"
9,4,5,Annealing,NaN,"100, 1800","C, sec"



===== NiOx synthesis (raw, as-is) =====


,LayerID,StepOrder,ProcessType,Descriptions,Parameter,Unit,Remarks
0,2,1,Dissolve,Ni(NO3)2·6H2O,6,M,NiOx synthesis
1,2,2,pH adjustment,add NaOH for pH10,7,M,NiOx synthesis
2,2,3,Wash,De-ionized water,4,times,NiOx synthesis
3,2,4,Dry,NaN,"80,12","C, hr",NiOx synthesis
4,2,5,Annealing,NaN,"270,3","C, hr",NiOx synthesis


## Re-schema —  normalized

ทุกตารางเชื่อมกันด้วยคีย์เดียว **`device_id`** (= ชื่อไฟล์ เช่น `10_1126_science_ade9637_Sample2`).

In [62]:
# เก็บข้อมูลดิบจากทุกไฟล์ (pass เดียว อ่านทุก sheet ต่อไฟล์) — ยังไม่ประกอบเป็นตาราง
# cell ถัดไปแต่ละอันจะประกอบ+แสดงตารางทีละตัว ให้รันดูผลลัพธ์ได้ทีละ table
devices, layers, vendors, materials, process, properties, stability = (
    [], [], [], [], [], [], [])
warnings = []

for f in files:
    device_id, doi_raw, sample_no = parse_device_id(f)
    devices.append({"device_id": device_id, "doi_raw": doi_raw,
                    "sample_no": sample_no, "source_file": os.path.basename(f)})
    xl = pd.ExcelFile(f)

    lay = _clean(pd.read_excel(f, "Layer")).dropna(how="all")
    lay.insert(0, "device_id", device_id)
    layers.append(lay)

    ven = _clean(pd.read_excel(f, "Vender")).dropna(how="all")
    ven.insert(0, "device_id", device_id)
    vendors.append(ven)

    mat = _clean(pd.read_excel(f, "Materials")).dropna(how="all")
    mat.insert(0, "device_id", device_id)
    materials.append(mat)

    # Process + any extra process-like sheet (e.g. "NiOx synthesis").
    # `source_sheet` keeps provenance so folded-in steps stay traceable.
    for sh in xl.sheet_names:
        if sh == "Process" or (sh not in KNOWN_SHEETS
                               and _is_process_like(pd.read_excel(f, sh, nrows=0).columns)):
            pro = _clean(pd.read_excel(f, sh)).dropna(how="all")
            pro.insert(0, "source_sheet", sh)
            pro.insert(0, "device_id", device_id)
            process.append(pro)

    prp = _clean(pd.read_excel(f, "Properties")).dropna(how="all")
    prp.insert(0, "device_id", device_id)
    properties.append(prp)

    stb = pd.read_excel(f, "Stability")
    for c in stb.columns:
        if str(c).strip().lower() not in STABILITY_MAP:
            warnings.append(f"{device_id} [Stability]: unmapped column "
                            f"'{str(c).strip()}' (dropped)")
    stb.columns = [STABILITY_MAP.get(str(c).strip().lower(), str(c).strip())
                   for c in stb.columns]
    stb = _clean(stb).dropna(how="all")
    stb.insert(0, "device_id", device_id)
    stability.append(stb)

    # flag any sheet we neither know nor recognized as process-like
    for sh in xl.sheet_names:
        if sh not in KNOWN_SHEETS and not _is_process_like(
                pd.read_excel(f, sh, nrows=0).columns):
            warnings.append(f"{device_id}: unknown sheet '{sh}' (ignored)")

if warnings:
    print("⚠️  SCHEMA DRIFT (data preserved where possible):")
    for w in warnings:
        print("   -", w)

def cat(frames):
    return pd.concat(frames, ignore_index=True) if frames else pd.DataFrame()

print("collect raw data —", len(files), "ไฟล์")

collect raw data — 44 ไฟล์


In [63]:
devices = pd.DataFrame(devices)
print(devices.shape)
display(devices.head(6))

(44, 4)


,device_id,doi_raw,sample_no,source_file
0,10.1002admi.202500231_Sample1,10.1002admi.202500231,1,10.1002admi.202500231_Sample1.xlsx
1,10.1002admi.202500231_Sample2,10.1002admi.202500231,2,10.1002admi.202500231_Sample2.xlsx
2,10.1002admi.202500231_Sample3,10.1002admi.202500231,3,10.1002admi.202500231_Sample3.xlsx
3,10.1002admi.202500231_Sample4,10.1002admi.202500231,4,10.1002admi.202500231_Sample4.xlsx
4,10.1016j.jechem.2021.11.029_Sample1,10.1016j.jechem.2021.11.029,1,10.1016j.jechem.2021.11.029_Sample1.xlsx
5,10.1016j.jechem.2021.11.029_Sample2,10.1016j.jechem.2021.11.029,2,10.1016j.jechem.2021.11.029_Sample2.xlsx


In [64]:
layers = cat(layers).rename(columns={
    "LayerID": "layer_id", "LayerType": "layer_type",
    "Material": "material", "Thickness (nm)": "thickness_nm"})
print(layers.shape)
display(layers.head(6))

(352, 5)


,device_id,layer_id,layer_type,material,thickness_nm
0,10.1002admi.202500231_Sample1,1,Anode,ITO,NaN
1,10.1002admi.202500231_Sample1,2,HTL,NiOx NPs,NaN
2,10.1002admi.202500231_Sample1,3,HTL treatment,NaN,NaN
3,10.1002admi.202500231_Sample1,4,PSK,[ (Cs.05FA.79MA.16)Pb(I.87Br.13)3 ],400-450
4,10.1002admi.202500231_Sample1,5,PSK treatment,NaN,NaN
5,10.1002admi.202500231_Sample1,6,ETL,C60,50


In [65]:
vendors = cat(vendors).rename(columns={
    "Material": "material", "%Purity": "purity", "Vender": "vendor"})
print(vendors.shape)
display(vendors.head(6))

(644, 4)


,device_id,material,purity,vendor
0,10.1002admi.202500231_Sample1,ITO coated glass,9 - 15 Ω □-1,"Colorado Concept Coatings LLC, 96041"
1,10.1002admi.202500231_Sample1,DMF,99.9,Sigma-Aldrich
2,10.1002admi.202500231_Sample1,DMSO,99,Alfa Aesar
3,10.1002admi.202500231_Sample1,CB,99.8,Acros Organics
4,10.1002admi.202500231_Sample1,C60,99,MER Corporation
5,10.1002admi.202500231_Sample1,BCP,96,Sigma Aldrich


In [66]:
materials = cat(materials).rename(columns={
    "LayerID": "layer_id", "Material": "material", "Type": "type",
    "Solvent": "solvent", "Amount": "amount", "Unit": "unit"})
print(materials.shape)
display(materials.head(6))

(543, 7)


,device_id,layer_id,material,type,solvent,amount,unit
0,10.1002admi.202500231_Sample1,1.0,ITO,NaN,NaN,NaN,NaN
1,10.1002admi.202500231_Sample1,2.0,NiOx NPs,Solution,De-ionized water,22.0,mg/ml
2,10.1002admi.202500231_Sample1,3.0,NaN,NaN,NaN,NaN,NaN
3,10.1002admi.202500231_Sample1,4.0,PbI2,Solution,DMF/DMSO (4/1 V/V),324.0,mg
4,10.1002admi.202500231_Sample1,4.0,CsI,Solution,DMF/DMSO (4/1 V/V),11.0,mg
5,10.1002admi.202500231_Sample1,4.0,PbBr2,Solution,DMF/DMSO (4/1 V/V),47.0,mg


In [67]:
process = cat(process).rename(columns={
    "LayerID": "layer_id", "StepOrder": "step_order",
    "ProcessType": "process_type", "Descriptions": "description",
    "Parameter": "parameter", "Unit": "unit", "Remarks": "note"})
print(process.shape)
display(process.head(6))

(702, 9)


,device_id,source_sheet,layer_id,step_order,process_type,description,parameter,unit,note
0,10.1002admi.202500231_Sample1,Process,1,1.0,Wash,"TritonX solution, Sonication, Acetone, Isoprop...",1200,sec,NaN
1,10.1002admi.202500231_Sample1,Process,1,2.0,PlasmaTreat,O2,155600600,"W, mtorr, sec",NaN
2,10.1002admi.202500231_Sample1,Process,2,1.0,Sonication,NaN,3600,sec,NaN
3,10.1002admi.202500231_Sample1,Process,2,2.0,SpinCoating,NaN,"3000, 30","RPM, sec",NaN
4,10.1002admi.202500231_Sample1,Process,2,3.0,Annealing,NaN,"270, 10800","C, sec",NaN
5,10.1002admi.202500231_Sample1,Process,4,1.0,Magnetic Stir,NaN,"60, 3600","C, sec",NaN


In [68]:
properties = cat(properties).rename(columns={
    "Properties": "property", "Values": "value",
    "Errors": "error", "Unit": "unit", "Remarks": "remarks"})

# Add "error" column if missing
if "error" not in properties.columns:
    properties["error"] = pd.NA
print(properties.shape)
display(properties.head(10))

(563, 6)


,device_id,property,value,error,unit,remarks
0,10.1002admi.202500231_Sample1,Voc,1.020,0.010,V,AM 1.5G
1,10.1002admi.202500231_Sample1,Jsc,20.970,1.130,mA cm−2,AM 1.5G
2,10.1002admi.202500231_Sample1,FF,0.760,0.005,NaN,AM 1.5G
3,10.1002admi.202500231_Sample1,PCE,16.310,1.000,%,AM 1.5G
4,10.1002admi.202500231_Sample1,Rs,0.680,NaN,Ohm/cm2,NaN
5,10.1002admi.202500231_Sample1,FWHM,0.144,0.079,eV,NaN
6,10.1002admi.202500231_Sample1,HTL optical band gap,3.850,NaN,eV,NiOx NPs
7,10.1002admi.202500231_Sample1,PSK optical band gap,1.609,NaN,eV,NaN
8,10.1002admi.202500231_Sample1,VBM,-5.720,NaN,eV,UPS
9,10.1002admi.202500231_Sample1,TRPL t1,6.000,1.000,ns,HTL/PSK half-stack


In [69]:
stability = cat(stability).reindex(columns=["device_id"] + STABILITY_COLS)
print(stability.shape)
display(stability.head(6))

(62, 10)


,device_id,environment,encapsulation,temperature_c,humidity_pct,time_hr,retention_pct,t80_hr,initial_pce,final_pce
0,10.1002admi.202500231_Sample1,ISOS-L2,N,NaN,NaN,72,NaN,NaN,16.31,14.5
1,10.1002admi.202500231_Sample2,ISOS-L2,N,NaN,NaN,72,NaN,NaN,17.11,10.5
2,10.1002admi.202500231_Sample3,ISOS-L2,N,NaN,NaN,72,NaN,NaN,16.97,11.8
3,10.1002admi.202500231_Sample4,ISOS-L2,N,NaN,NaN,72,NaN,NaN,16.25,13.4
4,10.1016j.jechem.2021.11.029_Sample1,Nitrogen (N2),N,NaN,NaN,2880,86,NaN,NaN,NaN
5,10.1016j.jechem.2021.11.029_Sample1,Operational (MPP),N,NaN,NaN,700,66,NaN,NaN,NaN


In [70]:
# รวมทุกตารางเป็น dict เดียว ให้ cell write_csv / display ด้านล่างใช้ต่อ
tables = {"devices": devices, "layers": layers, "vendors": vendors,
          "materials": materials, "process": process,
          "properties": properties, "stability": stability}
for name, df in tables.items():
    print(f"{name:12s} {df.shape}")

devices      (44, 4)
layers       (352, 5)
vendors      (644, 4)
materials    (543, 7)
process      (702, 9)
properties   (563, 6)
stability    (62, 10)


In [71]:
# เขียนตาราง normalized ออก processed/normalize/*.csv — source of truth ให้ eda_prepare.ipynb
def write_csv(tables):
    d = os.path.join(OUT_DIR, "normalize")
    os.makedirs(d, exist_ok=True)
    for name, df in tables.items():
        df.to_csv(os.path.join(d, f"{name}.csv"), index=False)

write_csv(tables)
print("wrote CSVs ->", os.path.join(OUT_DIR, "normalize"))

wrote CSVs -> processed\normalize


In [72]:
for name in ["devices","layers","materials","process","properties","stability"]:
    print(f"\n───── {name} ─────")
    display(tables[name].head(6))


───── devices ─────


,device_id,doi_raw,sample_no,source_file
0,10.1002admi.202500231_Sample1,10.1002admi.202500231,1,10.1002admi.202500231_Sample1.xlsx
1,10.1002admi.202500231_Sample2,10.1002admi.202500231,2,10.1002admi.202500231_Sample2.xlsx
2,10.1002admi.202500231_Sample3,10.1002admi.202500231,3,10.1002admi.202500231_Sample3.xlsx
3,10.1002admi.202500231_Sample4,10.1002admi.202500231,4,10.1002admi.202500231_Sample4.xlsx
4,10.1016j.jechem.2021.11.029_Sample1,10.1016j.jechem.2021.11.029,1,10.1016j.jechem.2021.11.029_Sample1.xlsx
5,10.1016j.jechem.2021.11.029_Sample2,10.1016j.jechem.2021.11.029,2,10.1016j.jechem.2021.11.029_Sample2.xlsx



───── layers ─────


,device_id,layer_id,layer_type,material,thickness_nm
0,10.1002admi.202500231_Sample1,1,Anode,ITO,NaN
1,10.1002admi.202500231_Sample1,2,HTL,NiOx NPs,NaN
2,10.1002admi.202500231_Sample1,3,HTL treatment,NaN,NaN
3,10.1002admi.202500231_Sample1,4,PSK,[ (Cs.05FA.79MA.16)Pb(I.87Br.13)3 ],400-450
4,10.1002admi.202500231_Sample1,5,PSK treatment,NaN,NaN
5,10.1002admi.202500231_Sample1,6,ETL,C60,50



───── materials ─────


,device_id,layer_id,material,type,solvent,amount,unit
0,10.1002admi.202500231_Sample1,1.0,ITO,NaN,NaN,NaN,NaN
1,10.1002admi.202500231_Sample1,2.0,NiOx NPs,Solution,De-ionized water,22.0,mg/ml
2,10.1002admi.202500231_Sample1,3.0,NaN,NaN,NaN,NaN,NaN
3,10.1002admi.202500231_Sample1,4.0,PbI2,Solution,DMF/DMSO (4/1 V/V),324.0,mg
4,10.1002admi.202500231_Sample1,4.0,CsI,Solution,DMF/DMSO (4/1 V/V),11.0,mg
5,10.1002admi.202500231_Sample1,4.0,PbBr2,Solution,DMF/DMSO (4/1 V/V),47.0,mg


... (2 outputs left out)


───── properties ─────


,device_id,property,value,error,unit,remarks
0,10.1002admi.202500231_Sample1,Voc,1.020,0.010,V,AM 1.5G
1,10.1002admi.202500231_Sample1,Jsc,20.970,1.130,mA cm−2,AM 1.5G
2,10.1002admi.202500231_Sample1,FF,0.760,0.005,NaN,AM 1.5G
3,10.1002admi.202500231_Sample1,PCE,16.310,1.000,%,AM 1.5G
4,10.1002admi.202500231_Sample1,Rs,0.680,NaN,Ohm/cm2,NaN
5,10.1002admi.202500231_Sample1,FWHM,0.144,0.079,eV,NaN



───── stability ─────


,device_id,environment,encapsulation,temperature_c,humidity_pct,time_hr,retention_pct,t80_hr,initial_pce,final_pce
0,10.1002admi.202500231_Sample1,ISOS-L2,N,NaN,NaN,72,NaN,NaN,16.31,14.5
1,10.1002admi.202500231_Sample2,ISOS-L2,N,NaN,NaN,72,NaN,NaN,17.11,10.5
2,10.1002admi.202500231_Sample3,ISOS-L2,N,NaN,NaN,72,NaN,NaN,16.97,11.8
3,10.1002admi.202500231_Sample4,ISOS-L2,N,NaN,NaN,72,NaN,NaN,16.25,13.4
4,10.1016j.jechem.2021.11.029_Sample1,Nitrogen (N2),N,NaN,NaN,2880,86,NaN,NaN,NaN
5,10.1016j.jechem.2021.11.029_Sample1,Operational (MPP),N,NaN,NaN,700,66,NaN,NaN,NaN
